In [1]:
# Clone ComfyUI fresh
%cd /content
!rm -rf /content/ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
%cd /content/ComfyUI

# Install deps
!pip install -r requirements.txt

# Download Juggernaut XL checkpoint (v8) used by your project defaults
!wget -c https://huggingface.co/lllyasviel/fav_models/resolve/main/fav/juggernautXL_v8Rundiffusion.safetensors -P /content/ComfyUI/models/checkpoints/

/content
Cloning into '/content/ComfyUI'...
remote: Enumerating objects: 35817, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 35817 (delta 22), reused 14 (delta 14), pack-reused 35779 (from 2)
Receiving objects: 100% (35817/35817), 84.12 MiB | 19.63 MiB/s, done.
Resolving deltas: 100% (24287/24287), done.
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.9/21.9 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.2/85.2 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 

In [2]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

torch: 2.10.0+cu128
cuda available: True
cuda version: 12.8
device: Tesla T4


In [7]:
import os, signal
os.kill(server.pid, signal.SIGTERM)  # stop server

In [8]:
%cd /content/ComfyUI
import subprocess, os

server_log = "/content/comfyui.log"
if os.path.exists(server_log):
    os.remove(server_log)

server = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    stdout=open(server_log, "w"),
    stderr=subprocess.STDOUT,
)

print("ComfyUI PID:", server.pid)

/content/ComfyUI
ComfyUI PID: 12842


In [9]:
%cd /content/ComfyUI
import subprocess, os, time

log = "/content/comfyui.log"
if os.path.exists(log):
    os.remove(log)

p = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    stdout=open(log, "w"),
    stderr=subprocess.STDOUT,
)

time.sleep(6)
print("PID:", p.pid, "alive:", p.poll() is None)
print(open(log, "r", encoding="utf-8", errors="ignore").read()[-2500:])

/content/ComfyUI
PID: 12851 alive: True
Found comfy_kitchen backend cuda: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'scaled_mm_nvfp4']}
Found comfy_kitchen backend triton: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8']}
Found comfy_kitchen backend eager: {'available': True, 'disabled': False, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'dequantize_mxfp8', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'scaled_mm_mxfp8', 'scaled_mm_nvfp4']}
Checkpoint files will always be loaded safely.
Total VRAM 14913 MB, total RAM 12976 MB
pytorch version

In [10]:
import time, requests

ok = False
for i in range(60):
    try:
        r = requests.get("http://127.0.0.1:8188", timeout=2)
        if r.status_code in (200, 404):
            ok = True
            print("ComfyUI ready, status:", r.status_code)
            break
    except Exception:
        pass
    time.sleep(1)

print("ready:", ok)

ComfyUI ready, status: 200
ready: True


In [12]:
# Download and make cloudflared executable
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

import subprocess, re, time, os

log = "/content/cloudflared.log"
if os.path.exists(log):
    os.remove(log)

p = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
    stdout=open(log, "w"),
    stderr=subprocess.STDOUT,
)

url = None
for _ in range(60):
    time.sleep(1)
    txt = open(log, "r", encoding="utf-8", errors="ignore").read()
    m = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", txt)
    if m:
        url = m.group(0)
        break

print("COMFYUI_URL =", url if url else "not found yet")

--2026-04-06 21:25:34--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64 [following]
--2026-04-06 21:25:34--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/731ab2f8-6b77-4adb-a7b3-1104525e9d72?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-06T22%3A04%3A43Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-04-